### Git

[Video](https://youtu.be/yyhOOgA2yfE) - [Video2](https://youtu.be/DRM2XoXlZqI) - [Cheat Sheet](https://git-scm.com/cheat-sheet)

Wenn wir ein Programm entwickeln, wollen wir oft verschiedene Entwicklungsstände aufheben. Mit Dateinamen wird das
schnell unübersichtlich:

<img src='bild1.png' width='300'>

**Git** ist ein Programm zur **Versionsverwaltung**. Es speichert auf Wunsch den Stand aller Dateien eines Ordners,
merkt sich, wer wann was geändert hat, und erlaubt, zu jedem früheren Stand zurückzukehren. Außerdem kann man an
mehreren Versionen gleichzeitig arbeiten und sie später zusammenführen. Fast alle Softwareprojekte werden heute mit
Git verwaltet.

#### Installation

Wir laden Git von https://git-scm.com/ herunter und installieren es. Die meisten Einstellungen lassen wir, bei der
Integration in den Windows-Explorer entfernen wir den Haken. Git-Befehle geben wir in der Windows-Eingabeaufforderung
oder im Terminal von VS Code ein, Git Bash und Git GUI brauchen wir nicht.

<img src='bild2.png' width='401'>

Ohne weitere Einstellung heißt der erste Branch in Git *master*, auf GitHub heißt er *main*. Damit beide gleich
heißen, wählen wir für Git *main*:

<img src='default_branch.png' width='401'>

Wir folgen der Empfehlung für den Aufruf von Git:

<img src='git_command_line.png' width='401'>

In der Eingabeaufforderung prüfen wir die Installation mit `git --version`:

<img src='git_version.png' width='500'>

#### Globale Einstellungen

Einmal nach der Installation geben wir Namen und E-Mail-Adresse an. Sie werden in jedem Commit gespeichert, damit man
bei der Zusammenarbeit sieht, wer was geändert hat.

```
git config --global user.name "Vorname Nachname"
git config --global user.email name@example.com
```

Wer bei der Installation den Branch-Namen nicht umgestellt hat, kann das nachholen:

```
git config --global init.defaultBranch main
```

### Git in diesem Notebook ausprobieren

In der Eingabeaufforderung tippt man die Befehle direkt ein, z.B. `git status`. Damit wir alles hier im Notebook
ausprobieren können, rufen wir Git über eine kleine Python-Funktion auf: `git('status')` entspricht dem Befehl
`git status`. (Mit `!git status` ginge es auch, aber dann erscheinen die Meldungen von Git nicht immer in der
richtigen Reihenfolge.)

Damit kein echtes Projekt durcheinanderkommt, legen wir ein Übungs-Repository in einem temporären Ordner an. Dateien
schreiben und anzeigen wir mit Python-Funktionen, das entspricht dem Bearbeiten im Editor. Git muss installiert sein,
in Google Colab ist es das schon.

In [1]:
import os
import shlex
import subprocess
import tempfile
from pathlib import Path

def git(befehl):
    '''führt einen Git-Befehl aus und zeigt alle Meldungen in der richtigen Reihenfolge'''
    erg = subprocess.run(['git'] + shlex.split(befehl), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         encoding='utf-8', errors='replace')
    print(erg.stdout, end='')

ordner = Path(tempfile.mkdtemp()) / 'GitDemo'
ordner.mkdir()
os.chdir(ordner)

def schreibe(datei, text):
    Path(datei).parent.mkdir(parents=True, exist_ok=True)
    Path(datei).write_text(text, encoding='utf-8')

def zeige(datei):
    if Path(datei).exists():
        print(Path(datei).read_text(encoding='utf-8'), end='')
    else:
        print(datei, 'gibt es nicht')

#### git init

`git init` macht den aktuellen Ordner zu einem **Repository** (kurz Repo). Git legt dafür einen versteckten Unterordner
`.git` an, in dem es alle Stände speichert. `-b main` gibt dem ersten Branch den Namen *main*, `-q` (quiet) spart die
Meldung. Name und E-Mail setzen wir hier nur für dieses Übungs-Repo (ohne `--global`).

In [2]:
git('init -q -b main')
git('config user.name "Demo"')
git('config user.email "demo@example.com"')
git('config advice.detachedHead false')

### Szenario 1: Stände sichern

Wir arbeiten an einer Datei `f1.txt` und sichern von Zeit zu Zeit den Stand. Git unterscheidet drei Bereiche:

<img src='g_bereiche.png' width='620'>

- **Arbeitsverzeichnis** (working directory): die Dateien, wie sie gerade im Ordner liegen.
- **Staging Area** (Index): der *Warenkorb* für den nächsten Commit. Mit `git add` legt man Änderungen hinein.
- **Repository**: die gesicherten Stände, die **Commits**. `git commit` macht aus dem Inhalt der Staging Area einen
  neuen Commit.

So kann man genau auswählen, welche Änderungen in einen Commit kommen. Die grünen Pfeile gehen zurück: `git restore`
holt eine Datei aus der Staging Area ins Arbeitsverzeichnis, `git restore --staged` setzt sie in der Staging Area auf
den Stand des letzten Commits zurück.

In [3]:
schreibe('f1.txt', '1\n')
git('status')

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	f1.txt

nothing added to commit but untracked files present (use "git add" to track)


`f1.txt` ist *untracked*: Git kennt die Datei noch nicht. Wir legen sie in die Staging Area und machen den ersten
Commit. Mit `-m` gibt man die **Commit-Nachricht** an, die beschreibt, was geändert wurde.

In [4]:
git('add f1.txt')
git('status --short')
git('commit -m "c1"')

A  f1.txt


[main (root-commit) aa5c121] c1
 1 file changed, 1 insertion(+)
 create mode 100644 f1.txt


`git status --short` zeigt den Status knapp: `A` heißt *added*, die Datei ist in der Staging Area. Wir ändern die Datei
zweimal und sichern jedes Mal. `git add .` legt alle geänderten Dateien des Ordners in die Staging Area.

In [5]:
schreibe('f1.txt', '1\n2\n')
git('add .')
git('commit -q -m "c2"')
schreibe('f1.txt', '1\n2\n3\n')
git('add .')
git('commit -q -m "c3"')
git('log --oneline --decorate')

6c58e6a (HEAD -> main) c3
cff19db c2
aa5c121 c1


`git log` zeigt die Commits, der neueste steht oben. Jeder Commit hat einen **Hash**, eine Prüfsumme aus seinem
Inhalt, Autor, Zeit und Vorgänger. Die ersten 7 Stellen reichen, um ihn anzusprechen. (Die Hashes sind bei jedem
Durchlauf andere, weil die Zeit eingeht.)

Ein Commit speichert einen vollständigen Schnappschuss aller Dateien, nicht nur die Änderungen. Unveränderte Dateien
legt Git dabei nicht doppelt ab, sondern verweist auf die schon gespeicherte Fassung. Deshalb bleibt ein Repo trotzdem
klein.

<img src='g_linear.png' width='330'>

**main** ist ein **Branch**, ein Zeiger auf den neuesten Commit. **HEAD** zeigt auf den Branch, auf dem wir gerade
arbeiten. Mit `HEAD~1` meint man den Commit vor HEAD, mit `HEAD~2` den davor.

#### Änderungen zurücknehmen: git restore

Wir ändern die Datei und überlegen es uns anders:

In [6]:
schreibe('f1.txt', '1\n2\n3\n4\n')
git('status --short')
git('restore f1.txt')
zeige('f1.txt')

 M f1.txt
1
2
3


` M` heißt: geändert, aber nicht in der Staging Area. `git restore f1.txt` holt den Stand aus der Staging Area zurück,
die Änderung ist weg.

Liegt die Änderung schon in der Staging Area, nimmt `git restore --staged` sie dort heraus. Die Datei im
Arbeitsverzeichnis bleibt dabei geändert:

In [7]:
schreibe('f1.txt', '1\n2\n3\n4\n')
git('add f1.txt')
git('status --short')
git('restore --staged f1.txt')
git('status --short')
zeige('f1.txt')

M  f1.txt
 M f1.txt
1
2
3
4


In [8]:
git('restore f1.txt')
zeige('f1.txt')

1
2
3


| Situation | Befehl | Status danach |
|---|---|---|
| Datei geändert, nicht gestaged | `git restore f1.txt` | sauber, Änderung verworfen |
| Datei geändert und gestaged | `git restore --staged f1.txt` | geändert, nicht gestaged |
| Datei geändert und gestaged | `git restore --staged --worktree f1.txt` | sauber, Änderung verworfen |

Vorsicht: Mit `git restore` verworfene Änderungen sind wirklich weg, sie waren ja nie in einem Commit.

#### Frühere Stände ansehen

Mit `git diff` sieht man Unterschiede. `git diff HEAD~2` vergleicht das Arbeitsverzeichnis mit dem Stand von c1:

In [9]:
git('diff HEAD~2')

diff --git a/f1.txt b/f1.txt
index d00491f..01e79c3 100644
--- a/f1.txt
+++ b/f1.txt
@@ -1 +1,3 @@
 1
+2
+3


Zeilen mit `+` gibt es nur im Arbeitsverzeichnis, Zeilen mit `-` nur im alten Stand. Eine einzelne Datei aus einem
früheren Commit holt man mit `--source` zurück:

In [10]:
git('restore --source=HEAD~1 f1.txt')
zeige('f1.txt')
git('status --short')

1
2
 M f1.txt


Jetzt ist `f1.txt` im Arbeitsverzeichnis auf dem Stand von c2. Möchte man diesen Stand behalten, macht man einen neuen
Commit. Wir nehmen es wieder zurück:

In [11]:
git('restore f1.txt')
zeige('f1.txt')

1
2
3


#### Detached HEAD

Man kann HEAD auch direkt auf einen früheren Commit setzen und sich den ganzen Stand ansehen. HEAD zeigt dann nicht
mehr auf einen Branch, sondern auf einen Commit: Er ist *detached* (abgetrennt).

<img src='g_detached.png' width='330'>

In [12]:
git('switch --detach HEAD~1')
zeige('f1.txt')
git('status')

HEAD is now at cff19db c2


1
2
HEAD detached at cff19db
nothing to commit, working tree clean


In [13]:
git('switch main')
zeige('f1.txt')

Previous HEAD position was cff19db c2
Switched to branch 'main'
1
2
3


Statt `git switch --detach HEAD~1` findet man oft den älteren Befehl `git checkout HEAD~1`, statt `git switch main`
auch `git checkout main`. `git switch -` springt zum zuletzt benutzten Branch zurück.

#### git stash

Mit ungesicherten Änderungen kann man nicht in jeden anderen Stand wechseln, sie könnten überschrieben werden.
`git stash` legt die Änderungen beiseite (stash: verstauen), danach ist das Arbeitsverzeichnis sauber. `git stash pop`
holt sie zurück.

In [14]:
schreibe('f1.txt', '1\n2\n3\n4\n')
git('stash')
zeige('f1.txt')

Saved working directory and index state WIP on main: 6c58e6a c3


1
2
3


In [15]:
git('stash pop')
zeige('f1.txt')
git('restore f1.txt')

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   f1.txt

no changes added to commit (use "git add" and/or "git commit -a")
Dropped refs/stash@{0} (c73c878f0989dc14336a660fc9f5d3668896e67e)


1
2
3
4


### Szenario 2: Branches

Wir haben einen Stand, der benutzt wird und den wir nicht durch Experimente unbrauchbar machen wollen. Trotzdem wollen
wir darauf aufbauend ein neues Feature entwickeln. Dafür gibt es **Branches** (Zweige).

`git switch -c feature` legt einen neuen Branch *feature* an (create) und wechselt hinein. Die folgenden Commits
kommen auf *feature*, *main* bleibt, wo es war.

In [16]:
git('switch -c feature')
schreibe('f2.txt', 'a\n')
git('add .')
git('commit -q -m "d1"')
schreibe('f2.txt', 'a\nb\n')
git('add .')
git('commit -q -m "d2"')
git('log --oneline --decorate --graph --all')

Switched to a new branch 'feature'


* ee90945 (HEAD -> feature) d2
* 03fafa5 d1
* 6c58e6a (main) c3
* cff19db c2
* aa5c121 c1


<img src='g_branch.png' width='420'>

`git log --graph --all` zeigt alle Branches. Auf *main* gibt es `f2.txt` noch nicht:

In [17]:
git('switch main')
zeige('f2.txt')
git('branch')

Switched to branch 'main'
f2.txt gibt es nicht
  feature
* main


`git branch` listet die Branches, der Stern markiert den aktuellen. Einen Branch kann man auch an einem früheren
Commit beginnen lassen: `git switch -c name HEAD~2` oder mit dem Hash des Commits.

#### Merge: Fast-Forward

Das Feature ist fertig und soll in *main*. Seit dem Abzweigen gab es auf *main* keine neuen Commits. Dann muss Git nur
den Zeiger *main* nach vorne schieben, es entsteht kein neuer Commit. Das heißt **Fast-Forward-Merge**.

In [18]:
git('merge feature')
git('log --oneline --decorate --graph --all')

Updating 6c58e6a..ee90945
Fast-forward
 f2.txt | 2 ++
 1 file changed, 2 insertions(+)
 create mode 100644 f2.txt
* ee90945 (HEAD -> main, feature) d2
* 03fafa5 d1
* 6c58e6a c3
* cff19db c2
* aa5c121 c1


<img src='g_ff.png' width='420'>

#### Merge mit Merge-Commit

Jetzt arbeiten wir auf beiden Seiten weiter: auf einem neuen Branch *readme* an einer neuen Datei, auf *main* an
`f1.txt`.

In [19]:
git('switch -q -c readme')
schreibe('README.md', 'Demo-Projekt\n')
git('add .')
git('commit -q -m "r1"')
git('switch -q main')
schreibe('f1.txt', '1\n2\n3\n4\n')
git('add .')
git('commit -q -m "c4"')
git('log --oneline --decorate --graph --all')

* 0c409be (HEAD -> main) c4
| * 11c8de0 (readme) r1
|/  
* ee90945 (feature) d2
* 03fafa5 d1
* 6c58e6a c3
* cff19db c2
* aa5c121 c1


Die beiden Branches haben sich auseinanderentwickelt. Beim Merge erzeugt Git einen **Merge-Commit** mit zwei
Vorgängern, der beide Änderungen enthält. `--no-edit` übernimmt die vorgeschlagene Commit-Nachricht.

In [20]:
git('merge --no-edit readme')
git('log --oneline --decorate --graph --all')

Merge made by the 'ort' strategy.
 README.md | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 README.md
*   9a45c52 (HEAD -> main) Merge branch 'readme'
|\  
| * 11c8de0 (readme) r1
* | 0c409be c4
|/  
* ee90945 (feature) d2
* 03fafa5 d1
* 6c58e6a c3
* cff19db c2
* aa5c121 c1


Allgemein sieht das so aus, c5 ist der Merge-Commit:

<img src='g_merge.png' width='450'>

#### Merge-Konflikte

Haben beide Branches dieselbe Stelle verschieden geändert, kann Git nicht entscheiden, was gelten soll: Es gibt einen
**Konflikt**.

In [21]:
git('switch -q -c experiment')
schreibe('f1.txt', '1\n2\n3\n4\n5\n')
git('commit -q -am "e1"')
git('switch -q main')
schreibe('f1.txt', '1\n2\n3\n4\nfünf\n')
git('commit -q -am "c6"')
git('merge experiment')

Auto-merging f1.txt
CONFLICT (content): Merge conflict in f1.txt
Automatic merge failed; fix conflicts and then commit the result.


(`git commit -am` macht `add` und `commit` in einem Schritt, aber nur für Dateien, die Git schon kennt.)

Git hat die Datei mit **Konfliktmarkierungen** versehen:

In [22]:
zeige('f1.txt')

1
2
3
4
<<<<<<< HEAD
fünf
5
>>>>>>> experiment


Zwischen `<<<<<<< HEAD` und `=======` steht die Fassung des aktuellen Branches (*current*), zwischen `=======` und
`>>>>>>> experiment` die des anderen (*incoming*). VS Code bietet dafür Knöpfe wie *Accept Current Change* an. Wir
entscheiden uns von Hand, entfernen die Markierungen und schließen den Merge mit einem Commit ab:

In [23]:
schreibe('f1.txt', '1\n2\n3\n4\n5\n')
git('add f1.txt')
git('commit -q --no-edit')
git('log --oneline --decorate --graph -6')

*   8a3982c (HEAD -> main) Merge branch 'experiment'
|\  
| * b5227eb (experiment) e1
* | a0bbebc c6
|/  
*   9a45c52 Merge branch 'readme'
|\  
| * 11c8de0 (readme) r1
* | 0c409be c4
|/  


Einen Merge, bei dem man sich verrannt hat, kann man mit `git merge --abort` abbrechen.

#### Rebase

Auch mit **Rebase** kann man die Änderungen zweier Branches zusammenführen. Statt eines Merge-Commits werden die
Commits des Feature-Branches neu auf den aktuellen Stand von *main* gesetzt:

<img src='g_rebase_vorher.png' width='380'> &nbsp;&nbsp; $\longrightarrow$ &nbsp;&nbsp;
<img src='g_rebase_nachher.png' width='420'>

d1' und d2' haben denselben Inhalt wie d1 und d2, sind aber neue Commits mit anderen Hashes. Die Geschichte wird
dadurch geradlinig.

In [24]:
git('switch -q -c rebase-demo HEAD~2')
schreibe('f3.txt', 'x\n')
git('add .')
git('commit -q -m "x1"')
git('log --oneline --decorate --graph --all -8')

*   8a3982c (main) Merge branch 'experiment'
|\  
| * b5227eb (experiment) e1
* | a0bbebc c6
|/  
| * 69739e5 (HEAD -> rebase-demo) x1
|/  
*   9a45c52 Merge branch 'readme'
|\  
| * 11c8de0 (readme) r1
* | 0c409be c4
|/  
* ee90945 (feature) d2


In [25]:
git('rebase main')
git('log --oneline --decorate --graph --all -8')

Rebasing (1/1)
Successfully rebased and updated refs/heads/rebase-demo.
* c5448a2 (HEAD -> rebase-demo) x1
*   8a3982c (main) Merge branch 'experiment'
|\  
| * b5227eb (experiment) e1
* | a0bbebc c6
|/  
*   9a45c52 Merge branch 'readme'
|\  
| * 11c8de0 (readme) r1
* | 0c409be c4
|/  
* ee90945 (feature) d2


Weil Rebase Commits durch neue ersetzt, soll man es nur für Commits benutzen, die noch niemand anders hat. Bei einem
Branch, den schon andere heruntergeladen haben, passt deren Geschichte sonst nicht mehr zu unserer. Auch beim Rebase
kann es Konflikte geben, die man lösen muss.

Danach ist der Merge in *main* ein Fast-Forward:

In [26]:
git('switch -q main')
git('merge rebase-demo')

Updating 8a3982c..c5448a2
Fast-forward
 f3.txt | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 f3.txt


#### git cherry-pick

Mit `cherry-pick` holt man einen einzelnen Commit aus einem anderen Branch und wendet ihn auf den aktuellen Branch an,
z.B. eine Fehlerkorrektur, ohne den ganzen Branch zu übernehmen.

In [27]:
git('switch -q -c bugfix')
schreibe('f4.txt', 'Fehler behoben\n')
git('add .')
git('commit -q -m "fix"')
schreibe('f4.txt', 'Fehler behoben\nnoch nicht fertig\n')
git('commit -q -am "fix2"')
git('switch -q main')
git('cherry-pick bugfix~1')
zeige('f4.txt')

[main e0d2838] fix
 Date: Sun Nov 8 15:25:47 2026 +0100
 1 file changed, 1 insertion(+)
 create mode 100644 f4.txt
Fehler behoben


### Commits rückgängig machen

#### git reset

`git reset` setzt den aktuellen Branch auf einen früheren Commit zurück. Die späteren Commits gehören dann nicht mehr
zum Branch.

<img src='g_reset.png' width='330'>

- `git reset HEAD~1`: Die Änderungen des entfernten Commits bleiben im Arbeitsverzeichnis, nicht gestaged (eine neue
  Datei ist dann wieder *untracked*, `??`). Man kann sie z.B. anders aufteilen oder auf einem anderen Branch committen.
- `git reset --soft HEAD~1`: Die Änderungen bleiben und sind gestaged.
- `git reset --hard HEAD~1`: Auch das Arbeitsverzeichnis wird zurückgesetzt, die Änderungen sind verloren.

In [28]:
schreibe('f5.txt', 'Versuch\n')
git('add .')
git('commit -q -m "versuch"')
git('reset HEAD~1')
git('status --short')
git('log --oneline --decorate -3')

?? f5.txt


e0d2838 (HEAD -> main) fix
c5448a2 (rebase-demo) x1
8a3982c Merge branch 'experiment'


In [29]:
git('add .')
git('commit -q -m "versuch"')
git('reset --hard HEAD~1')
git('status --short')
zeige('f5.txt')

HEAD is now at e0d2838 fix
f5.txt gibt es nicht


#### git revert

`git revert` nimmt die Änderungen eines Commits zurück, indem es einen **neuen** Commit mit den umgekehrten Änderungen
anlegt. Die Geschichte bleibt erhalten.

<img src='g_revert.png' width='420'>

In [30]:
git('log --oneline --decorate -2')
git('revert --no-edit HEAD')
git('log --oneline --decorate -3')
zeige('f4.txt')

e0d2838 (HEAD -> main) fix
c5448a2 (rebase-demo) x1


[main 5a05586] Revert "fix"
 Date: Sun Nov 8 15:25:47 2026 +0100
 1 file changed, 1 deletion(-)
 delete mode 100644 f4.txt
5a05586 (HEAD -> main) Revert "fix"
e0d2838 fix
c5448a2 (rebase-demo) x1
f4.txt gibt es nicht


Der Unterschied ist wichtig, sobald andere mit dem Repo arbeiten: `reset` schreibt die Geschichte um, Commits, die
andere schon haben, verschwinden aus dem Branch. `revert` fügt nur einen Commit hinzu und ist deshalb für gemeinsam
genutzte Branches der richtige Weg.

Revert macht nur genau diesen einen Commit rückgängig. Man kehrt damit nicht zum Stand eines älteren Commits zurück,
spätere Commits bleiben wirksam.

### .gitignore

In einer Datei `.gitignore` im Repo stehen Muster für Dateien, die Git nicht verwalten soll, z.B. Logdateien, Passwörter
oder automatisch erzeugte Dateien.

```
# Kommentarzeilen beginnen mit #
*.log
geheim.txt
notizen/
```

- `*.log`: alle Dateien mit der Endung `.log`
- `geheim.txt`: diese Datei, egal in welchem Ordner
- `notizen/`: der ganze Ordner

Kommentare müssen in einer eigenen Zeile stehen. Ein `#` hinter einem Muster gehört zum Muster.

Vorlagen für viele Programmiersprachen gibt es auf https://github.com/github/gitignore.

In [31]:
schreibe('.gitignore', '# Kommentarzeilen beginnen mit #\n*.log\ngeheim.txt\nnotizen/\n')
schreibe('fehler.log', 'Protokoll\n')
schreibe('geheim.txt', 'Passwort\n')
schreibe('notizen/idee.txt', 'Idee\n')
schreibe('programm.py', 'print(1)\n')
git('status --short')

?? .gitignore
?? programm.py


Nur `.gitignore` und `programm.py` tauchen auf. `git check-ignore -v` zeigt, welche Regel eine Datei ausschließt:

In [32]:
git('check-ignore -v fehler.log geheim.txt notizen/idee.txt')

.gitignore:2:*.log	fehler.log
.gitignore:3:geheim.txt	geheim.txt
.gitignore:4:notizen/	notizen/idee.txt


`.gitignore` wirkt nur auf Dateien, die Git noch nicht verwaltet. Ist eine Datei schon in einem Commit, muss man sie
erst mit `git rm --cached datei` aus der Verwaltung nehmen (die Datei selbst bleibt im Ordner) und das committen.

In [33]:
git('add .')
git('commit -q -m "gitignore"')
schreibe('daten.csv', 'a,b\n')
git('add daten.csv')
git('commit -q -m "daten"')
schreibe('.gitignore', '# Kommentarzeilen beginnen mit #\n*.log\ngeheim.txt\nnotizen/\ndaten.csv\n')
schreibe('daten.csv', 'a,b\n1,2\n')
git('status --short')

 M .gitignore
 M daten.csv


Obwohl `daten.csv` jetzt in `.gitignore` steht, meldet Git die Änderung, weil die Datei schon verwaltet wird.

In [34]:
git('rm -q --cached daten.csv')
git('commit -q -am "daten.csv nicht mehr verwalten"')
schreibe('daten.csv', 'a,b\n1,2\n3,4\n')
git('status --short')

`git status` meldet nichts mehr: `daten.csv` wird nicht mehr verwaltet und steht in `.gitignore`.

### Zusammenfassung

| Befehl | Wirkung |
|---|---|
| `git init` | Ordner zum Repository machen |
| `git status`, `git log --oneline --decorate --graph --all` | Zustand und Geschichte ansehen |
| `git add datei`, `git add .` | Änderungen in die Staging Area legen |
| `git commit -m "Nachricht"` | Stand der Staging Area als Commit sichern |
| `git restore datei` | Änderung im Arbeitsverzeichnis verwerfen |
| `git restore --staged datei` | Datei aus der Staging Area nehmen |
| `git diff`, `git diff HEAD~1` | Unterschiede anzeigen |
| `git switch --detach HEAD~1`, `git switch main` | früheren Stand ansehen, zurück |
| `git stash`, `git stash pop` | Änderungen beiseitelegen, zurückholen |
| `git switch -c name`, `git switch name`, `git branch` | Branch anlegen, wechseln, auflisten |
| `git merge name` | anderen Branch in den aktuellen übernehmen |
| `git rebase main` | Commits des aktuellen Branches auf main neu aufsetzen |
| `git cherry-pick hash` | einzelnen Commit übernehmen |
| `git reset HEAD~1`, `git reset --hard HEAD~1` | Branch zurücksetzen (Geschichte ändert sich) |
| `git revert hash` | Commit durch einen neuen Commit rückgängig machen |
| `git rm --cached datei` | Datei aus der Verwaltung nehmen |